# Experimental: Improving Classification Accuracy

Current best: ~67% test accuracy. This notebook tries:
1. **Subject-level aggregation** (mean + std per subject before training)
2. **Multiple classifiers** (RF, SVM, Gradient Boosting, Logistic Regression)
3. **Multiple random splits** (average over 20 different 70/30 splits)

---

In [1]:
#CYMO_CSV = './ann.cymo_parkceleb_per_recording.csv'
#META_CSV = './cymo_parkceleb_per_recording_metadata.csv'
CYMO_CSV   = '../ParkCeleb/ann.cymo_parkceleb_per_recording.csv'
META_CSV   = './cymo_parkceleb_per_recording_metadata.csv'
FEAT_CSV = './mrmr_top10_features.csv'  # try different feature files here

In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             roc_auc_score, roc_curve, auc, confusion_matrix)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10, 'axes.titleweight': 'bold'})
print('OK')

OK


In [3]:
# Load recording-level data
cymo = pd.read_csv(CYMO_CSV)
meta = pd.read_csv(META_CSV)
features = pd.read_csv(FEAT_CSV)['feature'].tolist()

tid_col = 'TID' if 'TID' in cymo.columns else cymo.columns[0]
meta_tid = 'TID' if 'TID' in meta.columns else meta.columns[0]

df = cymo.merge(meta[[meta_tid, 'group', 'subject']].drop_duplicates(),
                left_on=tid_col, right_on=meta_tid, how='inner')
df['label'] = (df['group'] == 'PD').astype(int)

for col in features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        df[col] = 0.0

df[features] = df[features].fillna(df[features].median())

print(f'Recording-level: {len(df)} rows, {df["subject"].nunique()} subjects')

Recording-level: 143818 rows, 100 subjects


---
## 1 — Subject-level aggregation

Instead of training on individual recordings, compute **mean and std** of each  
feature across all recordings per subject. This gives one row per subject  
with 2× the features (mean + variability).

In [4]:
# Aggregate per subject: mean and std of each feature
agg_dict = {}
for f in features:
    agg_dict[f] = ['mean', 'std']
agg_dict['label'] = 'first'
agg_dict['group'] = 'first'

subj_df = df.groupby('subject').agg(agg_dict).reset_index()

# Flatten column names: feature_mean, feature_std
subj_df.columns = ['subject'] + \
    [f'{f}_{stat}' for f, stat in subj_df.columns[1:] if stat != '']

# Fix label and group columns (they got suffixed too)
label_col = [c for c in subj_df.columns if 'label' in c][0]
group_col = [c for c in subj_df.columns if 'group' in c][0]
subj_df = subj_df.rename(columns={label_col: 'label', group_col: 'group'})

# Fill NaN std (subjects with only 1 recording will have NaN std)
subj_df = subj_df.fillna(0)

# Feature columns for the aggregated data
agg_features = [c for c in subj_df.columns if c not in ('subject', 'label', 'group')]

print(f'Subject-level: {len(subj_df)} subjects, {len(agg_features)} features (mean + std)')
print(f'  CN: {(subj_df["label"]==0).sum()}  PD: {(subj_df["label"]==1).sum()}')

# Also keep a mean-only version
mean_features = [c for c in agg_features if c.endswith('_mean')]
print(f'  Mean-only features: {len(mean_features)}')
print(f'  Mean+Std features:  {len(agg_features)}')

Subject-level: 100 subjects, 48 features (mean + std)
  CN: 60  PD: 40
  Mean-only features: 24
  Mean+Std features:  48


---
## 2 — Multi-classifier, multi-split evaluation

Run 20 random 70/30 splits. For each split, train 4 classifiers.  
Report average and std of each metric across splits.

In [5]:
# ============================================================
# 2.  Systematic experiment
# ============================================================

def get_classifiers():
    """Return dict of classifiers to try."""
    return {
        'LR': LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=42),
        'SVM_rbf': SVC(kernel='rbf', class_weight='balanced', probability=True, C=1.0, random_state=42),
        'SVM_lin': SVC(kernel='linear', class_weight='balanced', probability=True, C=1.0, random_state=42),
        'RF': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=3,
                                     class_weight='balanced', random_state=42, n_jobs=-1),
        'GB': GradientBoostingClassifier(n_estimators=100, max_depth=3, min_samples_leaf=5,
                                         learning_rate=0.1, random_state=42),
    }


def evaluate_split(data_df, feature_list, seed):
    """Run one 70/30 split and evaluate all classifiers."""
    np.random.seed(seed)
    
    cn = data_df[data_df['label'] == 0].sample(frac=1, random_state=seed)
    pd_g = data_df[data_df['label'] == 1].sample(frac=1, random_state=seed)
    
    cn_split = int(len(cn) * 0.7)
    pd_split = int(len(pd_g) * 0.7)
    
    train = pd.concat([cn.iloc[:cn_split], pd_g.iloc[:pd_split]])
    test = pd.concat([cn.iloc[cn_split:], pd_g.iloc[pd_split:]])
    
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train[feature_list].values)
    X_te = scaler.transform(test[feature_list].values)
    y_tr = train['label'].values
    y_te = test['label'].values
    
    results = []
    for name, clf in get_classifiers().items():
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        y_prob = clf.predict_proba(X_te)[:, 1]
        
        results.append({
            'classifier': name,
            'seed': seed,
            'acc': accuracy_score(y_te, y_pred),
            'f1': f1_score(y_te, y_pred, zero_division=0),
            'auc': roc_auc_score(y_te, y_prob) if len(np.unique(y_te)) > 1 else 0.5,
            'sens': recall_score(y_te, y_pred, zero_division=0),
            'spec': recall_score(y_te, y_pred, pos_label=0, zero_division=0),
        })
    
    return results

In [ ]:
# ============================================================
# 2.1  Run experiments across 3 feature configurations
# ============================================================
N_SPLITS = 20

configs = [
    ('Recording-level (mean only)', df, features, 'rec_mean'),
    ('Subject-level (mean only)', subj_df, mean_features, 'subj_mean'),
    ('Subject-level (mean + std)', subj_df, agg_features, 'subj_meanstd'),
]

all_results = []

for config_name, data, feat_list, config_id in configs:
    print(f'\nRunning: {config_name} ({len(feat_list)} features, {len(data)} samples)...')
    
    for seed in range(N_SPLITS):
        try:
            res = evaluate_split(data, feat_list, seed=seed)
            for r in res:
                r['config'] = config_name
                r['config_id'] = config_id
            all_results.extend(res)
        except Exception as e:
            print(f'  Seed {seed} failed: {e}')

results_df = pd.DataFrame(all_results)
print(f'\nTotal experiments: {len(results_df)}')


Running: Recording-level (mean only) (24 features, 143818 samples)...


In [ ]:
# ============================================================
# 2.2  Summary table: mean ± std across splits
# ============================================================
summary = results_df.groupby(['config', 'classifier']).agg(
    acc_mean=('acc', 'mean'), acc_std=('acc', 'std'),
    f1_mean=('f1', 'mean'), f1_std=('f1', 'std'),
    auc_mean=('auc', 'mean'), auc_std=('auc', 'std'),
    sens_mean=('sens', 'mean'), spec_mean=('spec', 'mean'),
).round(3).reset_index()

summary = summary.sort_values(['config', 'auc_mean'], ascending=[True, False])

print(f'RESULTS: Mean ± Std over {N_SPLITS} random 70/30 splits')
print('=' * 100)
print(f'{"Config":<30s} {"Classifier":<10s} {"Acc":>12s} {"F1":>12s} {"AUC":>12s} {"Sens":>6s} {"Spec":>6s}')
print('─' * 100)

prev_config = ''
for _, r in summary.iterrows():
    if r['config'] != prev_config:
        if prev_config:
            print()
        prev_config = r['config']
    print(f'{r["config"]:<30s} {r["classifier"]:<10s} '
          f'{r["acc_mean"]:.3f}±{r["acc_std"]:.3f} '
          f'{r["f1_mean"]:.3f}±{r["f1_std"]:.3f} '
          f'{r["auc_mean"]:.3f}±{r["auc_std"]:.3f} '
          f'{r["sens_mean"]:>.3f} {r["spec_mean"]:>.3f}')

In [ ]:
# ============================================================
# 2.3  Find the best overall combination
# ============================================================
best = summary.sort_values('auc_mean', ascending=False).head(5)
print('\nTop 5 combinations by AUC:')
for i, (_, r) in enumerate(best.iterrows()):
    print(f'  {i+1}. {r["config"]} + {r["classifier"]}: '
          f'AUC={r["auc_mean"]:.3f}±{r["auc_std"]:.3f}  '
          f'Acc={r["acc_mean"]:.3f}±{r["acc_std"]:.3f}')

In [ ]:
# ============================================================
# 2.4  Visualisation: grouped bar chart
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, metric, label in [(axes[0], 'acc_mean', 'Accuracy'),
                           (axes[1], 'auc_mean', 'ROC AUC'),
                           (axes[2], 'f1_mean', 'F1 Score')]:
    pivot = summary.pivot(index='classifier', columns='config', values=metric)
    pivot.plot(kind='bar', ax=ax, alpha=0.8, edgecolor='white', width=0.8)
    
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_ylim(0.3, 1.0)
    ax.axhline(0.5, color='gray', ls=':', lw=0.8, alpha=0.5)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(axis='y', alpha=0.2)
    ax.tick_params(axis='x', rotation=0)

plt.suptitle(f'Classifier × Feature Config Comparison (mean over {N_SPLITS} splits)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 2.5  Box plots of AUC distribution across splits
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

# Create combined label
results_df['combo'] = results_df['config_id'] + '\n' + results_df['classifier']

# Order by median AUC
order = results_df.groupby('combo')['auc'].median().sort_values(ascending=False).index

colors_map = {'rec_mean': '#95A5A6', 'subj_mean': '#3498DB', 'subj_meanstd': '#E74C3C'}
box_colors = []
for combo in order:
    config_id = results_df[results_df['combo'] == combo]['config_id'].iloc[0]
    box_colors.append(colors_map.get(config_id, '#999'))

data_to_plot = [results_df[results_df['combo'] == c]['auc'].values for c in order]

bp = ax.boxplot(data_to_plot, labels=order, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5, label='Random')
ax.set_ylabel('AUC')
ax.set_title(f'AUC Distribution Across {N_SPLITS} Random Splits')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('experiment_auc_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 — Train the best configuration and save

In [ ]:
# ============================================================
# 3.  Train best config on 70% subjects, evaluate on 30%
# ============================================================

# Pick the best config from the summary
best_row = summary.sort_values('auc_mean', ascending=False).iloc[0]
best_config_name = best_row['config']
best_clf_name = best_row['classifier']

print(f'Best combination: {best_config_name} + {best_clf_name}')
print(f'  Mean AUC: {best_row["auc_mean"]:.3f} ± {best_row["auc_std"]:.3f}')

# Determine which data and features to use
best_config_id = summary.sort_values('auc_mean', ascending=False).iloc[0]
# Map back to actual data
config_map = {
    'Recording-level (mean only)': (df, features),
    'Subject-level (mean only)': (subj_df, mean_features),
    'Subject-level (mean + std)': (subj_df, agg_features),
}
best_data, best_feats = config_map[best_config_name]

# Final 70/30 split (seed=42 for reproducibility)
np.random.seed(42)
cn_data = best_data[best_data['label'] == 0].sample(frac=1, random_state=42)
pd_data = best_data[best_data['label'] == 1].sample(frac=1, random_state=42)
cn_split = int(len(cn_data) * 0.7)
pd_split = int(len(pd_data) * 0.7)

final_train = pd.concat([cn_data.iloc[:cn_split], pd_data.iloc[:pd_split]])
final_test = pd.concat([cn_data.iloc[cn_split:], pd_data.iloc[pd_split:]])

final_scaler = StandardScaler()
X_tr = final_scaler.fit_transform(final_train[best_feats].values)
X_te = final_scaler.transform(final_test[best_feats].values)
y_tr = final_train['label'].values
y_te = final_test['label'].values

final_clf = get_classifiers()[best_clf_name]
final_clf.fit(X_tr, y_tr)

y_pred = final_clf.predict(X_te)
y_prob = final_clf.predict_proba(X_te)[:, 1]

acc = accuracy_score(y_te, y_pred)
f1 = f1_score(y_te, y_pred, zero_division=0)
auc_val = roc_auc_score(y_te, y_prob)
sens = recall_score(y_te, y_pred, zero_division=0)
spec = recall_score(y_te, y_pred, pos_label=0, zero_division=0)

print(f'\nFinal test results:')
print(f'  Acc  = {acc:.3f}')
print(f'  F1   = {f1:.3f}')
print(f'  AUC  = {auc_val:.3f}')
print(f'  Sens = {sens:.3f}')
print(f'  Spec = {spec:.3f}')

# Save
joblib.dump({
    'model': final_clf,
    'scaler': final_scaler,
    'features': best_feats,
    'config': best_config_name,
    'classifier': best_clf_name,
    'aggregation': 'subject_level' if 'Subject' in best_config_name else 'recording_level',
    'base_features': features,  # original mRMR features (before mean/std)
}, 'best_model.joblib')
print(f'\nSaved: best_model.joblib')

In [ ]:
# ============================================================
# 3.1  Final visualisation
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
ax = axes[0]
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['CN','PD'], yticklabels=['CN','PD'],
            cbar=False, annot_kws={'size':16})
ax.set_title(f'{best_clf_name} + {best_config_name}\nAcc={acc:.2f} AUC={auc_val:.2f}')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# ROC
ax = axes[1]
fpr, tpr, _ = roc_curve(y_te, y_prob)
ax.plot(fpr, tpr, 'r-', lw=2, label=f'AUC={auc_val:.3f}')
ax.plot([0,1],[0,1], 'k--', alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve'); ax.legend(); ax.grid(alpha=0.2)

plt.suptitle('Best Model — Final Test Results', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('best_model_results.png', dpi=150, bbox_inches='tight')
plt.show()